# 06 — Phase 2 Sandbox Backtest

This notebook is intentionally independent from Phase 1 diagnostics. It performs no model fitting, does not rebuild Phase 1 reports, and consumes only the pinned frozen-candidate predictions. The strategy suite now includes bounded dynamic-exit, entry-quality, and per-variant forensic checks.

Sandbox results remain non-promotable until the preregistered Future-OOS gate passes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_BASE = '/content/drive/MyDrive/yeniBot'
CHECKPT_DIR = f'{DRIVE_BASE}/checkpoints'
REPORT_DIR = f'{DRIVE_BASE}/reports'
os.makedirs(CHECKPT_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

In [ ]:
import os, subprocess, sys
REPO_URL = 'https://github.com/umutergul74/yeniBot.git'
REPO_DIR = '/content/yenibot_repo'
REPO_BRANCH = os.environ.get('YENIBOT_REPO_BRANCH', 'codex/phase2-sandbox')
if os.path.exists(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', '-B', REPO_BRANCH, f'origin/{REPO_BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, REPO_DIR], check=True)
repo_commit = subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'], text=True).strip()
repo_branch = subprocess.check_output(['git', '-C', REPO_DIR, 'branch', '--show-current'], text=True).strip()
assert repo_branch == REPO_BRANCH, f'Expected {REPO_BRANCH}, found {repo_branch}'
sys.path.insert(0, REPO_DIR)
print('Repository branch:', repo_branch)
print('Repository commit:', repo_commit)
print('After a changed checkout, use Runtime -> Restart session before trusting imports.')

In [ ]:
# Phase 2 does not need the training stack (torch, hmmlearn, numba, etc.).
!pip install -q "pandas>=2.2" "pyarrow>=15.0" "PyYAML>=6.0"

In [ ]:
PHASE1_RUN_ID = None  # Keep None for the latest report with a pinned frozen manifest.
PHASE2_SPLIT = 'test'
RUN_ALL_COST_SCENARIOS = True
RUN_STRATEGY_SUITE = True  # Bounded exploratory exits + entry filters; no automatic winner selection.
print('No training will run. Frozen predictions only.')

In [ ]:
import json
import os
import shutil
import time
from pathlib import Path

import pandas as pd
from IPython.display import display
from yenibot.automation.phase2_sandbox import main as run_phase2_sandbox

experiment_reports = Path(REPORT_DIR) / 'experiments'
if PHASE1_RUN_ID:
    report_dir = experiment_reports / str(PHASE1_RUN_ID)
else:
    eligible = sorted(
        (
            path for path in experiment_reports.iterdir()
            if path.is_dir()
            and (path / 'frozen_candidate_manifest.json').exists()
            and (path / 'phase2_readiness.json').exists()
        ),
        key=lambda path: path.name,
    )
    if not eligible:
        raise FileNotFoundError('No Phase 1 report contains the frozen Phase 2 input contract. Run Notebook 05 once.')
    report_dir = eligible[-1]

phase2_dir = report_dir / 'phase2_sandbox'
args = [
    '--report-dir', str(report_dir),
    '--checkpoint-dir', str(CHECKPT_DIR),
    '--output-dir', str(phase2_dir),
    '--mode', 'sandbox',
    '--split', PHASE2_SPLIT,
]
if RUN_ALL_COST_SCENARIOS:
    args.append('--all-cost-scenarios')
if RUN_STRATEGY_SUITE:
    args.append('--strategy-suite')

started = time.perf_counter()
run_phase2_sandbox(args)
elapsed = time.perf_counter() - started

report_path = phase2_dir / 'phase2_sandbox_report.json'
payload = json.loads(report_path.read_text(encoding='utf-8'))
summary = payload.get('summary', {})
max_delay = summary.get('max_entry_delay_hours')
assert max_delay is None or float(max_delay) <= 1.5, 'Stale next-bar execution crossed a data gap'

print('Phase 1 source report:', report_dir)
print('Phase 2 output:', phase2_dir)
print(f'Backtest runtime: {elapsed:.2f} seconds')
print('Fit operations performed: 0')
print('Evidence status:', summary.get('evidence_status'))

cost_summary_path = phase2_dir / 'phase2_cost_scenario_summary.csv'
if cost_summary_path.exists():
    cost_summary = pd.read_csv(cost_summary_path)
    columns = [
        'cost_scenario', 'trade_count', 'hit_rate', 'profit_factor',
        'compounded_return', 'max_drawdown', 'final_equity',
        'skipped_stale_entry_count', 'data_gap_forced_close_count',
    ]
    display(cost_summary[[column for column in columns if column in cost_summary.columns]])
else:
    display(pd.DataFrame([summary]))

variant_summary_path = phase2_dir / 'phase2_strategy_variant_summary.csv'
if variant_summary_path.exists():
    variant_summary = pd.read_csv(variant_summary_path)
    variant_columns = [
        'strategy_id', 'cost_scenario', 'trade_count', 'hit_rate',
        'profit_factor', 'compounded_return', 'max_drawdown',
        'delta_compounded_return_vs_baseline', 'delta_profit_factor_vs_baseline',
        'dynamic_stop_activation_share', 'selection_allowed_on_current_test',
    ]
    print('Exploratory strategy family (display only; no winner is selected):')
    display(variant_summary[[column for column in variant_columns if column in variant_summary.columns]])

variant_forensics_path = phase2_dir / 'phase2_strategy_forensics_summary.csv'
if variant_forensics_path.exists():
    variant_forensics = pd.read_csv(variant_forensics_path)
    forensic_columns = [
        'strategy_id', 'trade_count', 'gross_edge_bps_per_trade',
        'cost_bps_per_trade', 'net_edge_bps_per_trade',
        'compounded_net_return', 'max_holding_exit_share',
        'bootstrap_probability_compounded_return_positive',
    ]
    print('Base-cost strategy forensics (display only; no winner is selected):')
    display(variant_forensics[[column for column in forensic_columns if column in variant_forensics.columns]])

forensics_path = phase2_dir / 'phase2_forensics_summary.json'
if forensics_path.exists():
    print('Baseline forensics:')
    print(json.dumps(json.loads(forensics_path.read_text(encoding='utf-8')), indent=2))

trades = pd.read_csv(phase2_dir / 'phase2_trade_ledger.csv')
print('Base-scenario trades:', len(trades))
display(trades.head(20))

local_bundle_base = Path('/content') / f'phase2_sandbox_bundle_{report_dir.name}'
local_bundle = Path(shutil.make_archive(
    str(local_bundle_base),
    'zip',
    root_dir=phase2_dir.parent,
    base_dir=phase2_dir.name,
))

def durable_copy(source, destination, attempts=5):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    for attempt in range(1, attempts + 1):
        temporary = destination.with_name(destination.name + '.tmp')
        try:
            shutil.copyfile(source, temporary)
            os.replace(temporary, destination)
            return destination
        except OSError:
            try:
                if temporary.exists():
                    temporary.unlink()
            except OSError:
                pass
            if attempt == attempts:
                raise
            time.sleep(float(attempt))

versioned_bundle = durable_copy(
    local_bundle,
    Path(REPORT_DIR) / f'phase2_sandbox_bundle_{report_dir.name}.zip',
)
latest_bundle = durable_copy(
    local_bundle,
    Path(REPORT_DIR) / 'phase2_latest_sandbox_bundle.zip',
)
print('Versioned Phase 2 bundle:', versioned_bundle)
print('Latest Phase 2 bundle:', latest_bundle)